# 01 - Data Wrangling

**AI Forensic Triage Tool: Predicting Shooting Incidents in Boston**  
**Capstone Project — DSE 6311**  
**Author**: Ricardo Orellana  

**Goal**: Load raw Boston crime data, clean it, engineer initial time and location features, and save processed files for downstream analysis.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Direct GitHub raw URL (team-friendly)
RAW_CSV_URL = "https://raw.githubusercontent.com/Rick-997/AI-Forensics-Boston-Capstone/main/data/raw/crime_incident_reports_2023_present.csv"

# Robust path detection
CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"✅ Files will be saved here: {DATA_PROCESSED.resolve()}")

# Load raw data
print("Loading Boston Crime CSV from GitHub...")
df = pd.read_csv(RAW_CSV_URL, parse_dates=["OCCURRED_ON_DATE"], low_memory=False)
print(f"Original shape: {df.shape}")

df["SHOOTING"] = df["SHOOTING"].astype(int)
print(f"Shooting rate: {df['SHOOTING'].mean():.4%}")

# Time features
df["hour"] = df["OCCURRED_ON_DATE"].dt.hour
df["is_weekend"] = df["OCCURRED_ON_DATE"].dt.dayofweek.isin([5, 6]).astype(int)
df["is_night"] = ((df["hour"] >= 20) | (df["hour"] <= 5)).astype(int)

# Circular encoding
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Location cleaning
df = df[(df["Lat"] > 42.2) & (df["Lat"] < 42.4) & 
        (df["Long"] > -71.2) & (df["Long"] < -71.0)]
df["DISTRICT"] = df["DISTRICT"].fillna("Unknown")

# Violent offense proxy
df["is_violent"] = (df["OFFENSE_CODE_GROUP"]
                    .fillna("")
                    .astype(str)
                    .str.contains("Assault|Robbery|Homicide|Murder", case=False, na=False)
                    .astype(int))

# Keep final columns
cols_keep = [
    "INCIDENT_NUMBER", "OFFENSE_CODE_GROUP", "DISTRICT", "Lat", "Long",
    "SHOOTING", "hour", "is_weekend", "is_night", "hour_sin", "hour_cos",
    "is_violent", "YEAR", "MONTH"
]
df_clean = df[cols_keep].copy()

df_clean.to_parquet(DATA_PROCESSED / "crimes_cleaned.parquet", index=False)
df_clean.to_csv(DATA_PROCESSED / "crime_features.csv", index=False)

print(f"✅ Cleaned data saved! New shape: {df_clean.shape}")
print(df_clean["SHOOTING"].value_counts(normalize=True))

✅ Files will be saved here: C:\Users\richa\Documents\GitHub\AI-Forensics-Boston-Capstone\data\processed
Loading Boston Crime CSV from GitHub...
Original shape: (254640, 17)
Shooting rate: 0.6802%
✅ Cleaned data saved! New shape: (239371, 14)
SHOOTING
0    0.992986
1    0.007014
Name: proportion, dtype: float64
